# SmartDs 2D XY Slice Plot (Starter)

This notebook reads an already-2D BATSRUS slice file and plots a **flat-shaded triangulated field** in the XY plane.

It reuses the library (`SmartDs`) and the file's native mesh connectivity (`sds.corners`) and does **not** resample.

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

from starwinds_analysis.smart_ds import SmartDs

plt.rcParams['figure.dpi'] = 120

## Choose a 2D File

By default this looks for a `z=0*.plt` slice in `sample_data/`.

In [ ]:
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'sample_data').exists():
    # If launched from somewhere else, fall back to this repo root.
    REPO_ROOT = Path('/Users/dagfev/Documents/starwinds/starwinds-analysis')

sample_data = REPO_ROOT / 'sample_data'
z0_candidates = sorted(sample_data.glob('z=0*.plt'))

if not z0_candidates:
    available = sorted(p.name for p in sample_data.glob('*.plt'))
    raise FileNotFoundError(
        'No z=0 .plt file found in sample_data. Available .plt files: ' + ', '.join(available)
    )

DATA_FILE = z0_candidates[0]
print('Using:', DATA_FILE)

sds = SmartDs.from_file(str(DATA_FILE))
print('Title:', sds.title)
print('Zone :', sds.zone)
print('Points:', len(sds.points))
print('Variables:', len(sds.variables))

In [ ]:
# Inspect available fields
list(sds.variables)[:40]

## Pick XY Coordinates and a Field to Color By

For an XY-plane slice, these are commonly `X [R]` and `Y [R]`.

In [ ]:
X_FIELD = 'X [R]'
Y_FIELD = 'Y [R]'
COLOR_FIELD = 'Rho [g/cm^3]'  # change as needed

x = np.asarray(sds.variable(X_FIELD), dtype=float)
y = np.asarray(sds.variable(Y_FIELD), dtype=float)
c = np.asarray(sds.variable(COLOR_FIELD), dtype=float)

if not (np.isfinite(x).all() and np.isfinite(y).all()):
    raise ValueError('Non-finite XY coordinates found; this starter expects a clean 2D slice mesh.')

finite_c = np.isfinite(c)
print('Finite field values:', int(finite_c.sum()), '/', c.size)

In [ ]:
import matplotlib.tri as mtri

corners = np.asarray(sds.corners, dtype=int)
if corners.ndim != 2 or corners.shape[1] not in (3, 4):
    raise ValueError(f'Expected triangular or quad connectivity, got shape {corners.shape}')

if corners.shape[1] == 4:
    triangles = np.vstack([
        corners[:, [0, 1, 2]],
        corners[:, [0, 2, 3]],
    ])
else:
    triangles = corners

tris = mtri.Triangulation(x, y, triangles)
print('Cells (quads):', corners.shape[0])
print('Triangles   :', triangles.shape[0])

In [ ]:
# Flat-shaded triangulated plot on the native 2D mesh (no resampling)
fig, ax = plt.subplots(figsize=(7, 6))

c_plot = np.ma.masked_invalid(c)
finite = np.isfinite(c)
use_log = finite.any() and np.nanmin(c[finite]) > 0
norm = LogNorm() if use_log else None

img = ax.tripcolor(tris, c_plot, shading='flat', cmap='viridis', norm=norm)
ax.triplot(tris, color='k', linewidth=0.15, alpha=0.15)

ax.set_aspect('equal')
ax.set_xlabel(X_FIELD)
ax.set_ylabel(Y_FIELD)
ax.set_title(COLOR_FIELD)
fig.colorbar(img, ax=ax, label=COLOR_FIELD)
plt.show()

## Optional: Try a Derived Field via `SmartDs`

If your environment has `griblet` recipes configured, you can request derived fields on demand.

In [ ]:
# Uncomment if you want derived BATSRUS fields (and griblet is installed/configured):
# sds.add_batsrus_graph()
# mach_a = sds.variable('M_A [none]')
# np.nanmin(mach_a), np.nanmax(mach_a)